In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf 
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from nested_pandas import NestedDtype

In [3]:
def test_lc(path):

    LC_COLUMN = "lc"

    catalog = read_hats(
        path,    
    ).map_partitions(
        lambda df: df.assign(
            **{LC_COLUMN: df[LC_COLUMN].astype(NestedDtype.from_pandas_arrow_dtype(df.dtypes[LC_COLUMN]))},
        )
    )

    with Client() as client:
        # display(client)
        df = catalog.compute()
    
    return df
path='/media3/majumder/dataset/cepheids/hats/zubercal_vcep'
df = test_lc(path)

In [ ]:

id_=1085440417799958669
data = pd.DataFrame(df[df.index==id_]["lc"].values[0])
data["band"].replace({"i": "#e8ce3c", "r": "#297560", "g": "#221442"}, inplace=True)

In [108]:
hid = [2214796789392420138,
        2215747173394139728,
        509779825764154648,
        595072387673099010,
        2214357841673687294,
        2213985306096277960,
        591476440637045108,
        422648561147798206,
        2211209620212458611,
        2210913752747011045]

In [ ]:
OFFSET = 58000.0
count=0
counter = 600
band_map = {
    'green': np.float64(3.676371655248864),
    'red': np.float64(3.803892557467166),
    'black': np.float64(3.8937079572375954)
}

# --- The Solution ---
mapped_labels = []
filenames = list()
path = f"/home/torsha/workplace/dataset/final_resampled_data_4/test/"
#
#
#
glob_pattern = os.path.join(path, '*', '*.record')
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
num_files_found = tf.data.experimental.cardinality(filenames)
print(f"Number of files found: {num_files_found.numpy()}")
#
#
#
dataset = tf.data.TFRecordDataset(filenames)
#
#
#
for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    #
    last_index = example.context.feature['last_index'].int64_list.value
    label_cls = example.context.feature['label'].bytes_list.value
    label_cls = label_cls[0].decode('utf-8')
    time = tf.convert_to_tensor(example.feature_lists.feature_list['dim_0'].feature[0].float_list.value , dtype=tf.float64) + OFFSET
    # time = example.feature_lists.feature_list['dim_0'].feature[0].float_list.value
    mag = example.feature_lists.feature_list['dim_1'].feature[0].float_list.value
    band_sorted = example.feature_lists.feature_list['dim_3'].feature[0].float_list.value
    # last_index = example.context.feature['last_index'].int64_list.value
    id_ = example.context.feature['id'].int64_list.value[0]
    if id_ == hid[9]:
        print("ID: ", id_)
        print("Label: ", label_cls)
        break
    count += 1

In [137]:
# Your input data and mapping dictionary
# band_sorted = np.array([3.67637157, 3.80389261, 3.89370799])
band_map = {
    'green': np.float64(3.676371655248864),
    'red': np.float64(3.803892557467166),
    'black': np.float64(3.8937079572375954)
}

# --- The Solution ---
mapped_labels = []
for value_to_map in band_sorted:
    found_match = False
    for label, target_value in band_map.items():
        # np.isclose() checks if the two numbers are close within a small tolerance
        if np.isclose(value_to_map, target_value):
            mapped_labels.append(label)
            found_match = True
            break # Move to the next value in band_sorted
    
    if not found_match:
        mapped_labels.append('Unknown') # Handle cases where no close match is found

result_array = np.array(mapped_labels)

# print("--- Readable Loop-based Solution ---")
# print(f"Original array: {band_sorted}")
# print(f"Mapped array:   {result_array}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.suptitle(f"ID: {id_} | Label: {label_cls}")
# plt.subplot(1, 2, 1)
# plt.scatter(time, mag, s=10)
# plt.subplot(1, 2, 2)
plt.scatter(time, mag, c=result_array, s=10)